In [0]:
# 1. Bezpieczne pobranie sekretów w locie (całkowicie niewidoczne w logach)
adls_key = dbutils.secrets.get(scope="maritime_kv", key="adls-bronze-key")
eh_connection_string = dbutils.secrets.get(scope="maritime_kv", key="eh-conn-string")

storage_account_name = "adlsmaritimegen2"

# 2. Dynamiczne uwierzytelnienie dla sterownika Azure Data Lake
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    adls_key
)

# 3. Konfiguracja odczytu Event Hubs (Kafka) z bezpiecznym łańcuchem
eh_namespace = "evhns-maritime-2026"
eh_topic = "eh-ais-stream"

kafka_options = {
    "kafka.bootstrap.servers": f"{eh_namespace}.servicebus.windows.net:9093",
    "subscribe": eh_topic,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{eh_connection_string}";',
    "startingOffsets": "latest"
}

# 4. Odczyt strumieniowy i transformacja z Kafka do Delta (Medallion - Bronze)
df_bronze_stream = spark.readStream.format("kafka").options(**kafka_options).load()

df_transformed = df_bronze_stream.selectExpr(
    "CAST(key AS STRING) as mmsi_key", 
    "CAST(value AS STRING) as json_payload", 
    "timestamp as ingestion_time"
)

bronze_output_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/ais_stream_raw"
checkpoint_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/_checkpoints/ais_stream"

query = (df_transformed.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start(bronze_output_path)
)

# Koniecznie musimy kazać Pythonowi poczekać, aż ten micro-batch się zakończy, 
# zanim Workflows uzna zadanie za skończone!
query.awaitTermination()

print("Czysty strumień uruchomiony. Dane płyną na prawdziwe ADLS Gen2 z chronioną autoryzacją!")